# BERTimbau aplicado ao corpus STIL 2023

Este notebook usa o modelo `neuralmind/bert-base-portuguese-cased` para analisar
os artigos presentes em `stil2023_articles (1).json`.

As atividades incluem:

- preparação e divisão dos artigos em blocos;
- embeddings contextuais de artigos e palavras;
- similaridade entre artigos;
- agrupamento e visualização dos documentos;
- previsão de palavras mascaradas;
- pseudo-perplexidade;
- ajuste opcional do BERTimbau ao corpus com *masked language modeling* (MLM);
- comparação entre o modelo original e o modelo ajustado.

> **Importante:** BERTimbau é um modelo de linguagem mascarada. Sua
> pseudo-perplexidade não é diretamente comparável à perplexidade dos modelos
> de bigramas e trigramas, que usam previsão sequencial.


## Etapa 1: instalar dependências

No Google Colab, ative uma GPU em **Ambiente de execução > Alterar tipo de
ambiente de execução > T4 GPU** antes de executar as células.


In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn seaborn


## Etapa 2: importar bibliotecas e configurar o ambiente

In [ ]:
import json
import math
import random
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from datasets import Dataset
from google.colab import files
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoModel,
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_LENGTH = 512
CHUNK_STRIDE = 64

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


## Etapa 3: carregar `stil2023_articles (1).json`

Selecione o arquivo quando o botão de upload aparecer. O notebook também aceita
o arquivo já presente em `/content`.


In [ ]:
JSON_NAME = "stil2023_articles (1).json"
json_path = Path("/content") / JSON_NAME

if not json_path.exists():
    uploaded = files.upload()
    if JSON_NAME in uploaded:
        json_path = Path("/content") / JSON_NAME
    else:
        json_path = Path("/content") / next(iter(uploaded))

with json_path.open(encoding="utf-8") as file:
    artigos = json.load(file)

print(f"Arquivo: {json_path.name}")
print(f"Quantidade de artigos: {len(artigos)}")
print("Campos disponíveis:", list(artigos[0]))


## Etapa 4: preparar os textos

In [ ]:
def limpar_texto(texto):
    texto = unicodedata.normalize("NFC", str(texto or ""))
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = re.sub(r"\S+@\S+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


registros = []
for indice, artigo in enumerate(artigos):
    texto = limpar_texto(
        artigo.get("artigo_completo")
        or artigo.get("artigo_completo_pt")
        or artigo.get("resumo")
    )
    if not texto:
        continue
    registros.append(
        {
            "id": indice,
            "titulo": artigo.get("titulo", f"Artigo {indice + 1}"),
            "idioma": artigo.get("idioma", "Não informado"),
            "texto": texto,
            "caracteres": len(texto),
        }
    )

df_artigos = pd.DataFrame(registros)
display(df_artigos[["id", "titulo", "idioma", "caracteres"]])


## Etapa 5: carregar o BERTimbau e dividir textos longos

O BERTimbau aceita no máximo 512 tokens por entrada. Cada artigo será dividido
em blocos sobrepostos. A sobreposição reduz a perda de contexto nas fronteiras.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
bertimbau = AutoModel.from_pretrained(MODEL_NAME).to(device)
bertimbau.eval()


def dividir_em_blocos(texto, max_length=MAX_LENGTH, stride=CHUNK_STRIDE):
    codificacao = tokenizer(
        texto,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
    )

    blocos = []
    for indice, input_ids in enumerate(codificacao["input_ids"]):
        bloco = {
            "input_ids": input_ids,
            "attention_mask": codificacao["attention_mask"][indice],
        }
        if "token_type_ids" in codificacao:
            bloco["token_type_ids"] = codificacao["token_type_ids"][indice]
        blocos.append(bloco)
    return blocos


df_artigos["quantidade_blocos"] = df_artigos["texto"].apply(
    lambda texto: len(dividir_em_blocos(texto))
)
display(df_artigos[["titulo", "quantidade_blocos"]])


## Etapa 6: gerar embeddings dos artigos

Para cada bloco, é calculada a média dos tokens válidos. O embedding do artigo
é a média dos embeddings de todos os seus blocos.


In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    mascara = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    soma = torch.sum(last_hidden_state * mascara, dim=1)
    divisor = torch.clamp(mascara.sum(dim=1), min=1e-9)
    return soma / divisor


@torch.inference_mode()
def embedding_artigo(texto):
    vetores = []
    for bloco in dividir_em_blocos(texto):
        input_ids = torch.tensor([bloco["input_ids"]], device=device)
        attention_mask = torch.tensor([bloco["attention_mask"]], device=device)
        saida = bertimbau(input_ids=input_ids, attention_mask=attention_mask)
        vetor = mean_pooling(saida.last_hidden_state, attention_mask)
        vetores.append(vetor.cpu())
    return torch.cat(vetores).mean(dim=0).numpy()


embeddings_artigos = np.vstack(
    [embedding_artigo(texto) for texto in df_artigos["texto"]]
)
np.save("embeddings_artigos_bertimbau.npy", embeddings_artigos)

print("Formato da matriz:", embeddings_artigos.shape)


## Etapa 7: encontrar artigos semanticamente semelhantes

In [ ]:
matriz_similaridade = cosine_similarity(embeddings_artigos)


def artigos_similares(indice, quantidade=5):
    ordem = np.argsort(matriz_similaridade[indice])[::-1]
    ordem = [item for item in ordem if item != indice][:quantidade]
    return pd.DataFrame(
        {
            "artigo_similar": df_artigos.iloc[ordem]["titulo"].values,
            "similaridade": matriz_similaridade[indice, ordem],
        }
    )


INDICE_ARTIGO = 0
print("Artigo consultado:")
print(df_artigos.iloc[INDICE_ARTIGO]["titulo"])
display(artigos_similares(INDICE_ARTIGO))


### Mapa de similaridade entre todos os artigos

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(
    matriz_similaridade,
    cmap="viridis",
    xticklabels=df_artigos["id"],
    yticklabels=df_artigos["id"],
)
plt.title("Similaridade de cosseno entre artigos - BERTimbau")
plt.xlabel("ID do artigo")
plt.ylabel("ID do artigo")
plt.tight_layout()
plt.show()


## Etapa 8: agrupar e visualizar os artigos

In [ ]:
NUM_GRUPOS = 4

kmeans = KMeans(n_clusters=NUM_GRUPOS, random_state=SEED, n_init=20)
df_artigos["grupo"] = kmeans.fit_predict(embeddings_artigos)

pca = PCA(n_components=2, random_state=SEED)
coordenadas = pca.fit_transform(embeddings_artigos)
df_artigos["pca_1"] = coordenadas[:, 0]
df_artigos["pca_2"] = coordenadas[:, 1]

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=df_artigos,
    x="pca_1",
    y="pca_2",
    hue="grupo",
    palette="tab10",
    s=100,
)
for _, linha in df_artigos.iterrows():
    plt.annotate(
        str(linha["id"]),
        (linha["pca_1"], linha["pca_2"]),
        xytext=(4, 4),
        textcoords="offset points",
    )
plt.title("Artigos STIL 2023 representados por embeddings BERTimbau")
plt.tight_layout()
plt.show()

display(
    df_artigos[["id", "grupo", "titulo"]].sort_values(["grupo", "id"])
)


## Etapa 9: prever palavras mascaradas

Use exatamente um marcador `[MASK]` na frase. O resultado representa as
palavras consideradas mais prováveis pelo BERTimbau naquele contexto.


In [ ]:
preenchedor = pipeline(
    "fill-mask",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1,
)

frase = "O processamento de linguagem natural utiliza modelos de [MASK]."
previsoes = preenchedor(frase, top_k=10)

pd.DataFrame(
    [
        {
            "token": item["token_str"].strip(),
            "probabilidade": item["score"],
            "frase_completa": item["sequence"],
        }
        for item in previsoes
    ]
)


## Etapa 10: comparar uma palavra em diferentes contextos

Esta atividade demonstra a principal diferença entre BERTimbau e Word2Vec:
a representação da palavra muda conforme a frase.


In [ ]:
@torch.inference_mode()
def embedding_palavra(frase, palavra, modelo=bertimbau):
    entradas = tokenizer(
        frase,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    offsets = entradas.pop("offset_mapping")[0].tolist()
    inicio = frase.casefold().find(palavra.casefold())
    if inicio < 0:
        raise ValueError(f"Palavra '{palavra}' não encontrada na frase.")
    fim = inicio + len(palavra)

    indices = [
        indice
        for indice, (token_inicio, token_fim) in enumerate(offsets)
        if token_fim > inicio and token_inicio < fim
    ]
    entradas = {chave: valor.to(device) for chave, valor in entradas.items()}
    saida = modelo(**entradas).last_hidden_state[0, indices]
    return saida.mean(dim=0).cpu().numpy()


palavra = "modelo"
frases = [
    "O modelo de linguagem foi treinado com textos científicos.",
    "O pesquisador apresentou um modelo matemático para o fenômeno.",
    "O novo modelo de carro foi apresentado ao mercado.",
]
vetores = np.vstack([embedding_palavra(frase, palavra) for frase in frases])

display(
    pd.DataFrame(
        cosine_similarity(vetores),
        index=[f"Contexto {i + 1}" for i in range(len(frases))],
        columns=[f"Contexto {i + 1}" for i in range(len(frases))],
    )
)
for indice, frase_contexto in enumerate(frases, start=1):
    print(f"{indice}. {frase_contexto}")


## Etapa 11: calcular pseudo-perplexidade

O algoritmo mascara cada token separadamente e mede a probabilidade atribuída
ao token original. Como o custo cresce com o número de tokens, a demonstração
usa sentenças ou trechos curtos.

> O valor abaixo é uma **pseudo-perplexidade de modelo mascarado**. Não use
> esse número como substituto direto das perplexidades de bigramas e trigramas.


In [ ]:
modelo_mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)
modelo_mlm.eval()


@torch.inference_mode()
def pseudo_perplexidade(texto, modelo=modelo_mlm, max_tokens=128):
    entradas = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=max_tokens,
    )
    input_ids = entradas["input_ids"][0]
    attention_mask = entradas["attention_mask"][0]
    especiais = set(tokenizer.all_special_ids)
    posicoes = [
        i for i, token_id in enumerate(input_ids.tolist())
        if token_id not in especiais
    ]
    perdas = []

    for posicao in posicoes:
        ids_mascarados = input_ids.clone()
        token_original = ids_mascarados[posicao].item()
        ids_mascarados[posicao] = tokenizer.mask_token_id

        saida = modelo(
            input_ids=ids_mascarados.unsqueeze(0).to(device),
            attention_mask=attention_mask.unsqueeze(0).to(device),
        )
        log_probs = F.log_softmax(saida.logits[0, posicao], dim=-1)
        perdas.append(-log_probs[token_original].item())

    return math.exp(float(np.mean(perdas))) if perdas else float("nan")


trechos_teste = [
    "A análise do corpus permite identificar padrões recorrentes da escrita científica.",
    "Os modelos de linguagem representam relações entre palavras e contextos.",
]
pd.DataFrame(
    {
        "texto": trechos_teste,
        "pseudo_perplexidade": [
            pseudo_perplexidade(texto) for texto in trechos_teste
        ],
    }
)


## Etapa 12: ajuste opcional ao corpus STIL 2023

Esta seção continua o pré-treinamento do BERTimbau com MLM. O corpus contém
apenas 30 artigos, portanto o objetivo é adaptação de domínio, não treinamento
do zero. Use poucas épocas e compare os resultados para observar possível
sobreajuste.

Esta etapa pode levar vários minutos e requer GPU.


In [ ]:
# Coloque EXECUTAR_AJUSTE = True para realizar o treinamento.
EXECUTAR_AJUSTE = False

if EXECUTAR_AJUSTE:
    textos_treino = []
    for texto in df_artigos["texto"]:
        for bloco in dividir_em_blocos(texto):
            textos_treino.append(
                tokenizer.decode(
                    bloco["input_ids"],
                    skip_special_tokens=True,
                )
            )

    dataset = Dataset.from_dict({"text": textos_treino})

    def tokenizar_lote(lote):
        return tokenizer(
            lote["text"],
            truncation=True,
            max_length=MAX_LENGTH,
            return_special_tokens_mask=True,
        )

    dataset_tokenizado = dataset.map(
        tokenizar_lote,
        batched=True,
        remove_columns=["text"],
    )
    dataset_tokenizado = dataset_tokenizado.train_test_split(
        test_size=0.1,
        seed=SEED,
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,
    )
    modelo_ajustado = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

    argumentos = TrainingArguments(
        output_dir="/content/bertimbau-stil2023",
        overwrite_output_dir=True,
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )

    treinador = Trainer(
        model=modelo_ajustado,
        args=argumentos,
        train_dataset=dataset_tokenizado["train"],
        eval_dataset=dataset_tokenizado["test"],
        data_collator=data_collator,
    )
    treinador.train()
    metricas = treinador.evaluate()
    print(metricas)
    print("Perplexidade MLM aproximada:", math.exp(metricas["eval_loss"]))

    modelo_ajustado.save_pretrained("/content/bertimbau-stil2023-final")
    tokenizer.save_pretrained("/content/bertimbau-stil2023-final")
else:
    print("Ajuste não executado. Altere EXECUTAR_AJUSTE para True quando desejar.")


## Etapa 13: comparar o modelo original com o ajustado

In [ ]:
if EXECUTAR_AJUSTE:
    modelo_ajustado = modelo_ajustado.to(device)
    modelo_ajustado.eval()

    comparacao = []
    for texto in trechos_teste:
        comparacao.append(
            {
                "texto": texto,
                "BERTimbau original": pseudo_perplexidade(
                    texto,
                    modelo=modelo_mlm,
                ),
                "BERTimbau ajustado": pseudo_perplexidade(
                    texto,
                    modelo=modelo_ajustado,
                ),
            }
        )
    display(pd.DataFrame(comparacao))
else:
    print("Execute primeiro a etapa de ajuste para gerar esta comparação.")


## Etapa 14: exportar resultados

São exportados os metadados dos grupos, a matriz de similaridade e os
embeddings dos artigos.


In [ ]:
df_artigos.drop(columns=["texto"]).to_csv(
    "artigos_bertimbau_grupos.csv",
    index=False,
    encoding="utf-8",
)
pd.DataFrame(
    matriz_similaridade,
    index=df_artigos["id"],
    columns=df_artigos["id"],
).to_csv("similaridade_artigos_bertimbau.csv", encoding="utf-8")
np.save("embeddings_artigos_bertimbau.npy", embeddings_artigos)

print("Arquivos gerados:")
print("- artigos_bertimbau_grupos.csv")
print("- similaridade_artigos_bertimbau.csv")
print("- embeddings_artigos_bertimbau.npy")


## Interpretação dos resultados

- **Similaridade:** valores próximos de 1 indicam representações semelhantes,
  mas não provam que os artigos têm o mesmo tema ou qualidade.
- **Agrupamento:** os grupos do K-Means são exploratórios e dependem do número
  de grupos escolhido.
- **PCA:** a projeção em duas dimensões perde parte da informação dos vetores.
- **Palavras contextuais:** a mesma palavra pode receber vetores diferentes em
  frases diferentes.
- **Pseudo-perplexidade:** deve ser usada para comparar versões compatíveis do
  BERTimbau, não para declarar superioridade sobre modelos de n-gramas.
- **Ajuste no corpus:** redução da perda no próprio corpus pode representar
  adaptação ao domínio ou sobreajuste. Uma avaliação externa seria necessária
  para uma conclusão geral.
